In [1]:
!pip install numpy pandas scikit-learn gdown

  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 90.6 MB/s eta 0:00:00a 0:00:01
Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Kiểm tra phiên bản
import numpy, pandas, sklearn
print(f"NumPy     : {numpy.__version__}")
print(f"Pandas    : {pandas.__version__}")
print(f"Sklearn   : {sklearn.__version__}")

NumPy     : 2.4.4
Pandas    : 3.0.2
Sklearn   : 1.8.0


In [3]:
# Tải file CSV từ Google Drive
!gdown 1jh2p2DlaWsDo_vEWIcTrNh3mUuXd-cw6
# Output: vi_text_retrieval.csv (download về thư mục hiện tại)

Downloading...
From: https://drive.google.com/uc?id=1jh2p2DlaWsDo_vEWIcTrNh3mUuXd-cw6
To: /workspace/generative_agents/buoi_2/vi_text_retrieval.csv
100%|████████████████████████████████████████| 201k/201k [00:00<00:00, 2.23MB/s]


In [4]:
import numpy as np
import pandas as pd

vi_data_df = pd.read_csv('vi_text_retrieval.csv')
vi_data_df.head()   # Xem 5 dòng đầu

,id,question,text
0,1570446247,Quang Hải giành được chức vô địch U21 quốc gia...,"Năm 2013 , Nguyễn Quang Hải giành chức vô địch..."
1,1570445661,Mỗi hiệp bóng đá kéo dài bao lâu,Một trận đấu bóng đá thông thường có hai hiệp ...
2,1570382095,Quân đội Hoa Kỳ gồm những lực lượng nào,Quân đội Hoa Kỳ hay Các lực lượng vũ trang Hoa...
3,1570382072,Ngọc Lan là ai,Ngọc Lan ( 28 tháng 12 năm 1956 - 6 tháng 3 20...
4,1570382037,Thu Phương từng được những giải thưởng nào,Cô được coi là một trong những ca sĩ thuộc thế...


In [5]:
vi_data_df.info()
# Xem: số dòng, tên cột, kiểu dữ liệu, null values
print(vi_data_df.shape)            # (số_dòng, số_cột)
print(vi_data_df.columns.tolist())  # ['text', ...]

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        500 non-null    int64
 1   question  500 non-null    str  
 2   text      500 non-null    str  
dtypes: int64(1), str(2)
memory usage: 11.8 KB
(500, 3)
['id', 'question', 'text']


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Bước 1: Lấy cột text và chuẩn hoá chữ thường
context = vi_data_df['text']
context = [doc.lower() for doc in context]

# Bước 2: Fit vectorizer trên toàn corpus
# → học vocabulary và tính IDF cho từng token
tfidf_vectorizer = TfidfVectorizer()
context_embedded = tfidf_vectorizer.fit_transform(context).toarray()
# Shape: (N_docs, vocab_size) — lưu vào RAM, dùng lại nhiều lần

print(f"Số tài liệu  : {context_embedded.shape[0]}")
print(f"Kích thước vocab: {context_embedded.shape[1]}")

# Xem thử vector của tài liệu đầu tiên
context_embedded[7][0]  # phần tử [7, 0] của ma trận

Số tài liệu  : 500
Kích thước vocab: 2913


np.float64(0.31126580760710637)

In [7]:
def tfidf_search(question, tfidf_vectorizer, context_embedded, top_d=5):
  """
  Tìm kiếm tài liệu liên quan dựa trên TF-IDF + Cosine Similarity.

  Args:
      question        : Câu hỏi hoặc query của người dùng
      tfidf_vectorizer: Vectorizer đã được fit() trên corpus
      context_embedded: Ma trận TF-IDF đã tính sẵn [N_docs × vocab]  ← THÊM MỚI
      top_d           : Số kết quả trả về

  Returns:
      List of dict {'id': int, 'cosine_score': float}
  """
  # Transform query thành vector TF-IDF (chỉ dùng transform, KHÔNG fit)
  query_embedded = tfidf_vectorizer.transform([question.lower()]).toarray()

  # ✅ FIX: Dùng lại context_embedded đã tính sẵn — không tính lại!
  cosine_scores = cosine_similarity(query_embedded, context_embedded)[0]

  # Sắp xếp giảm dần và lấy top_d kết quả
  results = []
  for idx in cosine_scores.argsort()[-top_d:][::-1]:
    results.append({
        'id': int(idx),
        'cosine_score': round(float(cosine_scores[idx]), 4)  # sửa typo: consine → cosine
    })
  return results

In [8]:
def corr_search(question, tfidf_vectorizer, context_embedded, top_d=5):
  """
  Tìm kiếm dựa trên Pearson Correlation thay vì Cosine Similarity.

  Pearson Correlation = cosine similarity sau khi trừ mean (centering).
  Ưu điểm: Loại bỏ bias về tần suất trung bình, nhạy hơn với sự phân kỳ.

  Args:
      question        : Câu hỏi hoặc query của người dùng
      tfidf_vectorizer: Vectorizer đã được fit()
      context_embedded: Ma trận TF-IDF đã tính sẵn [N_docs × vocab]
      top_d           : Số kết quả trả về
  """
  # Transform query
  query_vec = tfidf_vectorizer.transform([question.lower()]).toarray()[0]

  # ─── Pearson Correlation ───────────────────────────────────────
  # Centering: trừ mean để loại bỏ bias tần suất tổng thể
  query_centered   = query_vec   - query_vec.mean()
  context_centered = context_embedded - context_embedded.mean(axis=1, keepdims=True)

  # Tính dot product sau centering = numerator của Pearson
  numerator = context_centered @ query_centered

  # Norm của từng vector (để chuẩn hoá)
  norm_query   = np.linalg.norm(query_centered) + 1e-10   # tránh chia 0
  norm_context = np.linalg.norm(context_centered, axis=1) + 1e-10

  corr_scores = numerator / (norm_context * norm_query)
  # ───────────────────────────────────────────────────────────────

  results = []
  for idx in corr_scores.argsort()[-top_d:][::-1]:
    results.append({
        'id': int(idx),
        'corr_score': round(float(corr_scores[idx]), 4)
    })
  return results

In [9]:
# Test 1: Tìm tài liệu giống nhất với chính nó (expected: score ≈ 1.0)
question = vi_data_df.iloc[0]['text']
results  = tfidf_search(question, tfidf_vectorizer, context_embedded, top_d=5)
print(results)
# Kết quả mong đợi: id=0 với cosine_score=1.0 ở đầu danh sách

[{'id': 0, 'cosine_score': 1.0}, {'id': 384, 'cosine_score': 0.2047}, {'id': 136, 'cosine_score': 0.1967}, {'id': 88, 'cosine_score': 0.1924}, {'id': 10, 'cosine_score': 0.1684}]


In [10]:
# Test 2: Câu hỏi về bóng đá
question = "Quang Hải giành được chức vô định U21 năm nào?"
results  = tfidf_search(question, tfidf_vectorizer, context_embedded)
print("=== Kết quả tìm kiếm ===")
for r in results:
    print(f"[Score: {r['cosine_score']:.4f}] {vi_data_df.iloc[r['id']]['text'][:120]}...")
    print()

=== Kết quả tìm kiếm ===
[Score: 0.5372] Năm 2013 , Nguyễn Quang Hải giành chức vô địch U21 quốc gia 2013 cùng với đội trẻ Hà Nội T&T và tạo nên cú sốc khi trở t...

[Score: 0.1864] Tuyên Quang là một tỉnh thuộc vùng Đông Bắc Việt Nam . Có tỉnh lỵ là Thành phố Tuyên Quang ....

[Score: 0.1412] Dưới triều Nguyễn , năm 1822 ( Minh Mạng thứ 3 ) đổi tên trấn Sơn Nam Hạ thành trấn Nam Định . Đến năm Minh Mạng 13 ( 18...

[Score: 0.1260] Arsenal đã giành được 13 chức vô địch quốc gia , 13 Cúp FA ( kỷ lục ) , 2 Cúp Liên đoàn Anh , 1 Cúp Liên đoàn Thế kỷ , 1...

[Score: 0.1238] Thể chế thành phố được xác định theo quyết định của Chính phủ dựa trên một số tiêu chí nhất định như kích thước , dân số...



In [11]:
# Test 3: So sánh cosine vs pearson correlation
question = "Vua minh mạng là đời thứ mấy?"

print("─── TF-IDF Cosine Search ───")
results_cos = tfidf_search(question, tfidf_vectorizer, context_embedded)
for result in results_cos:
    answer = vi_data_df.iloc[result['id']].text
    print(f"[{result['cosine_score']:.4f}] {answer[:100]}")
    print()

print("─── Pearson Correlation Search ───")
results_corr = corr_search(question, tfidf_vectorizer, context_embedded)
for result in results_corr:
    answer = vi_data_df.iloc[result['id']].text
    print(f"[{result['corr_score']:.4f}] {answer[:100]}")
    print()

─── TF-IDF Cosine Search ───
[0.1796] Dưới triều Nguyễn , năm 1822 ( Minh Mạng thứ 3 ) đổi tên trấn Sơn Nam Hạ thành trấn Nam Định . Đến n

[0.1660] Nhà Tần chỉ tồn tại được 15 năm ( 221 - 206 TCN ) với 3 đời vua . Từ vua tới đại thần , không người 

[0.1409] Và cuối cùng , sau hàng thế kỉ ngóng đợi , bộ quốc sử " Đại Việt sử ký toàn thư " đã được in toàn bộ

[0.1318] Ngày 24 tháng 1 năm 1712 , Friedrich chào đời tại Berlin . Ông là con của vua Phổ Friedrich Wilhelm 

[0.1226] Tôn Trung Sơn, nguyên danh là Tôn Văn, tự Tải Chi, hiệu Nhật Tân, Dật Tiên là nhà cách mạng Trung Qu

─── Pearson Correlation Search ───
[0.1763] Dưới triều Nguyễn , năm 1822 ( Minh Mạng thứ 3 ) đổi tên trấn Sơn Nam Hạ thành trấn Nam Định . Đến n

[0.1619] Nhà Tần chỉ tồn tại được 15 năm ( 221 - 206 TCN ) với 3 đời vua . Từ vua tới đại thần , không người 

[0.1366] Và cuối cùng , sau hàng thế kỉ ngóng đợi , bộ quốc sử " Đại Việt sử ký toàn thư " đã được in toàn bộ

[0.1275] Ngày 24 tháng 1 năm 1712 , Friedrich ch